In [2]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 87.0 MB/s eta 0:00:00


In [9]:
import pennylane as qml
from pennylane import numpy as np

# Number of estimation qubits
n_estimation = 4

# Total wires
wires = list(range(n_estimation + 1))

dev = qml.device("default.qubit", wires=n_estimation + 1)

# Test with Pauli Z Hamiltonian
H = qml.PauliZ(n_estimation)

# Evolution time
t = np.pi / 4

# U = exp(-iHt)
U = qml.matrix(qml.exp(H, coeff=-1j * t))

@qml.qnode(dev)
def qpe():

    # Prepare eigenstate |1>
    qml.PauliX(wires=n_estimation)

    # Apply QPE
    qml.QuantumPhaseEstimation(
        U,
        target_wires=[n_estimation],
        estimation_wires=list(range(n_estimation))
    )

    return qml.probs(wires=range(n_estimation))

probs = qpe()

print("Measurement probabilities:")
for i, p in enumerate(probs):
    if p > 1e-4:
        print(f"{i:04b}: {p:.4f}")

phase_index = np.argmax(probs)
phase = phase_index / (2**n_estimation)

print("\nEstimated phase =", phase)
print(qml.draw(qpe, level="device")())

Measurement probabilities:
0010: 1.0000

Estimated phase = 0.125
0: ──H─╭●──────────────────────────────╭QFT†─┤ ╭Probs
1: ──H─│───────╭●──────────────────────├QFT†─┤ ├Probs
2: ──H─│───────│───────╭●──────────────├QFT†─┤ ├Probs
3: ──H─│───────│───────│───────╭●──────╰QFT†─┤ ╰Probs
4: ──X─╰U(M0)⁸─╰U(M0)⁴─╰U(M0)²─╰U(M0)¹───────┤       

M0 = 
[[0.70710678-0.70710678j 0.        +0.j        ]
 [0.        +0.j         0.70710678+0.70710678j]]


In [8]:
import pennylane as qml
from pennylane import numpy as np

# Heisenberg Hamiltonian
J = 1.0

H = (
    J * (qml.PauliX(0) @ qml.PauliX(1))
    + J * (qml.PauliY(0) @ qml.PauliY(1))
    + J * (qml.PauliZ(0) @ qml.PauliZ(1))
)

# Convert to matrix
H_matrix = qml.matrix(H)

print("Hamiltonian Matrix:")
print(H_matrix)

# Diagonalization
eigvals, eigvecs = np.linalg.eigh(H_matrix)

print("\nEigenvalues:")
print(eigvals)

print("\nGround state energy:")
print(eigvals[0])

print("\nGround state vector:")
print(eigvecs[:,0])

Hamiltonian Matrix:
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  2.+0.j  0.+0.j]
 [ 0.+0.j  2.+0.j -1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]

Eigenvalues:
[-3.  1.  1.  1.]

Ground state energy:
-3.0

Ground state vector:
[ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j]


In [6]:
E = 2 * np.pi * phase / t

print("Recovered energy =", E)

Recovered energy = 1.0
